In [ ]:
from figure_paths import DATA, RESULTS, WORK, FIGURES, setting
from pathlib import Path
from contextlib import ExitStack
from datetime import datetime, timezone
import hashlib
import json
import os
import shutil
import tempfile
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
from PIL import Image

RESULT_DIR = RESULTS
FIGURE_DIR = FIGURES
METRICS_DIR = RESULTS / 'best_model_spatial_figures'
GROUP_SUFFIX = {'B': '_residual', 'P': '_residual'}
FILE_OVERRIDES = {}
LEADS = tuple(range(1, 7))
COMPONENTS = ('u', 'v', 'w')       # w 文件按下方约定恢复为 omega
LEVELS = np.array([850., 925., 1000.])
GROUPS = tuple(GROUP_SUFFIX)
CONFIGS = [(g, h) for g in GROUPS for h in LEADS]
KINDS = ('testy', 'predicty')
METRIC_NAMES = ('R', 'RMSE', 'MAE', 'RMSPE')
W_TO_OMEGA = -10.0               # 本项目原始保存 w=-0.1*omega；仅转换一次
BLOCK_SIZE = 128                 # CPU 沿时间分块；不使用 GPU
TIME_ROUND_TOL_NS = 1000
NETCDF_TEMP_DIR = str(WORK)
WORK.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
REUSE_OWN_CACHE = False           # 只复用本 notebook 生成且签名一致的指标
CHECK_REPORTED_PERIOD = False
EXPECTED_COMMON_COUNT = 42070
EXPECTED_COMMON_START = '2019-03-15T02:00:00'
EXPECTED_COMMON_END = '2023-12-31T23:00:00'
SCHEMA_VERSION = 'best-model-spatial-v1'
FIG_DPI = 800                    # 高于600 dpi
FONT_SCALE = 1.0                 # 183 mm刊用宽度：默认刻度10.5 pt，轴标签11.5 pt
SHOW_FIGURES = True
METRICS_PATH = METRICS_DIR / 'best_model_grid_metrics.nc'
AUDIT_PATH = METRICS_DIR / 'input_audit.json'

ALIASES = {
    'time': ('time', 'valid_time', 'times', 'date', 'datetime'),
    'level': ('level', 'levels', 'pressure_level', 'isobaricInhPa', 'isobaricInPa', 'plev'),
    'latitude': ('latitude', 'lat'),
    'longitude': ('longitude', 'lon'),
}


def nc_path(group, lead, component, kind):
    source_kind = 'testy' if group == 'P' else kind
    key = ('B', int(lead), component, source_kind)
    return Path(FILE_OVERRIDES.get(key, RESULT_DIR / f'da_transunet_{source_kind}_{component}_{lead}_24year_changjiang_residual.nc'))


def rounded_hour_times(values, source):
    raw = np.asarray(values)
    if not np.issubdtype(raw.dtype, np.datetime64):
        raise TypeError(f'{source}: time须可解码为datetime64，不能将非标准日历静默转为标准日历。')
    raw = raw.astype('datetime64[ns]')
    if np.isnat(raw).any() or raw.size == 0:
        raise ValueError(f'{source}: 空时间轴或NaT。')
    ns = raw.astype(np.int64)
    if np.any(np.diff(ns) <= 0):
        raise ValueError(f'{source}: 原时间轴重复或倒序。')
    hour = 3_600_000_000_000
    quotient, remainder = np.divmod(ns, hour)
    rounded_ns = (quotient + (remainder >= hour // 2)) * hour
    delta = rounded_ns - ns
    if np.max(np.abs(delta)) > TIME_ROUND_TOL_NS:
        raise ValueError(f'{source}: 非整点偏差{np.max(np.abs(delta))} ns超过允许的编码噪声。')
    if np.any(np.diff(rounded_ns) <= 0):
        raise ValueError(f'{source}: 整点编码修正后出现重复或倒序。')
    changes = [dict(index=int(i), original=str(raw[i]), rounded=str(np.datetime64(int(rounded_ns[i]), 'ns')),
                    correction_ns=int(delta[i])) for i in np.flatnonzero(delta)]
    return rounded_ns.astype('datetime64[ns]'), changes


def canonical_field(ds, variable, source):
    if variable in ds.data_vars:
        da = ds[variable]
    else:
        candidates = []
        for name, value in ds.data_vars.items():
            if not np.issubdtype(value.dtype, np.number):
                continue
            if all(any(alias in value.dims for alias in aliases) for aliases in ALIASES.values()):
                candidates.append(name)
        if len(candidates) != 1:
            raise ValueError(f'{source}: 找不到{variable}且无法唯一确定四维风场，候选={candidates}。')
        da = ds[candidates[0]]
    rename = {}
    for standard, aliases in ALIASES.items():
        found = [a for a in aliases if a in da.dims]
        if len(found) != 1:
            raise ValueError(f'{source}: {standard}维度不存在或有歧义：{found}。')
        rename[found[0]] = standard
    da = da.rename({a: b for a, b in rename.items() if a != b})
    for dim in tuple(da.dims):
        if dim not in ALIASES:
            if da.sizes[dim] != 1:
                raise ValueError(f'{source}: 不能自动丢弃额外非单例维度{dim}。')
            da = da.isel({dim: 0}, drop=True)
    for dim in ALIASES:
        if dim not in da.coords or da[dim].ndim != 1:
            raise ValueError(f'{source}: {dim}必须有一维坐标。')
    lev = np.asarray(da.level.values, dtype=np.float64)
    units = str(da.level.attrs.get('units', '')).strip().lower().replace(' ', '')
    if units in ('pa', 'pascal', 'pascals') or (not units and np.all(lev > 2000)):
        lev = lev / 100.0
    elif units not in ('', 'hpa', 'hectopascal', 'hectopascals', 'mb', 'millibar', 'millibars'):
        raise ValueError(f'{source}: 未识别的气压层单位{units!r}。')
    if len(np.unique(lev)) != len(lev):
        raise ValueError(f'{source}: 气压层坐标重复。')
    indices = []
    for wanted in LEVELS:
        hit = np.flatnonzero(np.isclose(lev, wanted, rtol=0, atol=1e-8))
        if len(hit) != 1:
            raise ValueError(f'{source}: 缺少或重复气压层{wanted} hPa。')
        indices.append(int(hit[0]))
    da = da.assign_coords(level=lev).isel(level=indices)
    for dim in ('latitude', 'longitude'):
        c = np.asarray(da[dim].values, dtype=np.float64)
        if not np.isfinite(c).all() or len(np.unique(c)) != len(c):
            raise ValueError(f'{source}: {dim}坐标含非有限值或重复。')
        da = da.isel({dim: np.argsort(c)}).assign_coords({dim: np.sort(c)})
    da = da.transpose('time', 'level', 'latitude', 'longitude')
    return da


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(tmp, path)


def file_stat(path):
    stat = Path(path).stat()
    return dict(path=str(Path(path).resolve()), size=stat.st_size, mtime_ns=stat.st_mtime_ns)


def scan_inputs():
    records, missing = {}, []
    for group, lead in CONFIGS:
        for component in COMPONENTS:
            for kind in KINDS:
                path = nc_path(group, lead, component, kind)
                if not path.is_file():
                    missing.append(str(path))
                records[(group, lead, component, kind)] = dict(path=path)
    if missing:
        raise FileNotFoundError('缺少测试NC。请检查RESULT_DIR、GROUP_SUFFIX或FILE_OVERRIDES：\n' + '\n'.join(missing))
    common, reference_lat, reference_lon, audit = None, None, None, []
    for key, record in records.items():
        group, lead, component, kind = key
        path = record['path']
        with xr.open_dataset(path, chunks=None, cache=False) as ds:
            variable = f"{'testy' if group == 'P' else kind}_{component}"
            field = canonical_field(ds, variable, path)
            times, corrections = rounded_hour_times(field.time.values, path)
            if group == 'P' and kind == 'predicty':
                times = times + np.timedelta64(int(lead), 'h')
            latitude = np.asarray(field.latitude.values)
            longitude = np.asarray(field.longitude.values)
            if reference_lat is None:
                reference_lat, reference_lon = latitude, longitude
            elif not (np.array_equal(latitude, reference_lat) and np.array_equal(longitude, reference_lon)):
                raise ValueError(f'{path}: 空间网格与其他模型不同；不进行插值、近邻匹配或空间交集。')
            common = times if common is None else np.intersect1d(common, times, assume_unique=True)
            record.update(times=times, variable=variable, stat=file_stat(path))
            entry = dict(group=group, lead_hour=int(lead), component=component, kind=kind,
                         variable=str(field.name), count=int(len(times)),
                         time_start=str(times[0]), time_end=str(times[-1]),
                         time_corrections=corrections, **record['stat'])
            audit.append(entry)
    if common is None or len(common) < 3:
        raise ValueError('所有配置之间少于3个共同目标时次。')
    if np.any(np.diff(common) != np.timedelta64(1, 'h')):
        raise ValueError('共同目标时间不连续逐小时；请先核对数据，不插值或补造时次。')
    if CHECK_REPORTED_PERIOD:
        expected = (len(common) == EXPECTED_COMMON_COUNT and
                    common[0] == np.datetime64(EXPECTED_COMMON_START, 'ns') and
                    common[-1] == np.datetime64(EXPECTED_COMMON_END, 'ns'))
        if not expected:
            raise ValueError(f'实际共同期{len(common)}小时，{common[0]}至{common[-1]}，与已验收报告不同。请核对文件映射；不要直接裁剪或改成预期数。')
    for entry, record in zip(audit, records.values()):
        idx = np.searchsorted(record['times'], common)
        if not np.array_equal(record['times'][idx], common):
            raise AssertionError('时间索引映射失败。')
        record['indices'] = idx
        excluded = np.setdiff1d(record['times'], common, assume_unique=True)
        entry['excluded_count'] = int(len(excluded))
        entry['excluded_times'] = [str(t) for t in excluded]
    settings = dict(schema=SCHEMA_VERSION, persistence='own test reference at target minus lead, matched by timestamp', groups=GROUP_SUFFIX, leads=LEADS, components=COMPONENTS,
                    levels=LEVELS.tolist(), time_round_tolerance_ns=TIME_ROUND_TOL_NS,
                    w_to_omega=W_TO_OMEGA, common_count=int(len(common)),
                    common_time_sha256=hashlib.sha256(common.astype('<i8').tobytes()).hexdigest(),
                    finite_mask='all_groups_and_leads_per_component_level_grid_time',
                    spatial_mask='metric_specific_all_groups_and_leads', aggregation='equal_grid_then_equal_nine_units',
                    source_files=[record['stat'] for record in records.values()])
    signature = hashlib.sha256(json.dumps(settings, sort_keys=True, ensure_ascii=False).encode('utf-8')).hexdigest()
    payload = dict(settings=settings, input_signature=signature,
                   fingerprint_note='Source fingerprints use absolute path, size and mtime; not full source-content hashes.',
                   common_start=str(common[0]), common_end=str(common[-1]), files=audit)
    return records, common, reference_lat, reference_lon, signature, payload


In [ ]:
def merge_moments(old_n, old_mean_x, old_mean_y, old_m2_x, old_m2_y, old_cov,
                  x, y, valid):
    bn = valid.sum(axis=0).astype(np.int64)
    bx = np.divide(np.where(valid, x, 0.0).sum(axis=0), bn,
                   out=np.zeros(bn.shape, dtype=np.float64), where=bn > 0)
    by = np.divide(np.where(valid, y, 0.0).sum(axis=0), bn,
                   out=np.zeros(bn.shape, dtype=np.float64), where=bn > 0)
    dx = np.where(valid, x - bx, 0.0)
    dy = np.where(valid, y - by, 0.0)
    bm2x = np.sum(dx * dx, axis=0)
    bm2y = np.sum(dy * dy, axis=0)
    bcov = np.sum(dx * dy, axis=0)
    total = old_n + bn
    fraction = np.divide(bn, total, out=np.zeros(bn.shape, dtype=np.float64), where=total > 0)
    factor = np.divide(old_n.astype(np.float64) * bn, total,
                       out=np.zeros(bn.shape, dtype=np.float64), where=total > 0)
    shift_x = bx - old_mean_x
    shift_y = by - old_mean_y
    old_mean_x += shift_x * fraction
    old_mean_y += shift_y * fraction
    old_m2_x += bm2x + shift_x * shift_x * factor
    old_m2_y += bm2y + shift_y * shift_y * factor
    old_cov += bcov + shift_x * shift_y * factor
    return bn


def time_selection(indices):
    if len(indices) and (len(indices) == 1 or np.all(np.diff(indices) == 1)):
        return slice(int(indices[0]), int(indices[-1]) + 1)
    return indices


def assert_sources_unchanged():
    for record in records.values():
        if file_stat(record['path']) != record['stat']:
            raise RuntimeError(f"源文件在核对后发生变化：{record['path']}。请重新运行代码框1。")


def compute_grid_metrics():
    assert_sources_unchanged()
    shape = (len(GROUPS), len(LEADS), len(COMPONENTS), len(LEVELS), len(latitude), len(longitude))
    output = {name: np.full(shape, np.nan, dtype=np.float64) for name in METRIC_NAMES}
    ranges = np.full(shape, np.nan, dtype=np.float64)
    counts = np.zeros(shape[2:], dtype=np.int64)
    spatial_shape = shape[3:]
    cfg_shape = (len(CONFIGS),) + spatial_shape
    for ci, component in enumerate(COMPONENTS):
        n = np.zeros(spatial_shape, dtype=np.int64)
        mean_t, mean_p, m2_t, m2_p, cov, sse, sae = [np.zeros(cfg_shape, dtype=np.float64) for _ in range(7)]
        truth_min = np.full(cfg_shape, np.inf)
        truth_max = np.full(cfg_shape, -np.inf)
        scale = float(W_TO_OMEGA) if component == 'w' else 1.0
        if not np.isfinite(scale) or scale == 0:
            raise ValueError('W_TO_OMEGA必须是明确的非零有限转换系数。')
        with ExitStack() as stack:
            fields = {}
            for group, lead in CONFIGS:
                for kind in KINDS:
                    key = (group, lead, component, kind)
                    ds = stack.enter_context(xr.open_dataset(records[key]['path'], chunks=None, cache=False))
                    fields[(group, lead, kind)] = canonical_field(ds, records[key]['variable'], records[key]['path'])
            for start in range(0, len(common_times), BLOCK_SIZE):
                stop = min(start + BLOCK_SIZE, len(common_times))
                pairs = []
                valid = np.ones((stop-start,) + spatial_shape, dtype=bool)
                for group, lead in CONFIGS:
                    pair = []
                    for kind in KINDS:
                        key = (group, lead, component, kind)
                        indices = records[key]['indices'][start:stop]
                        values = np.asarray(fields[(group, lead, kind)].isel(time=time_selection(indices)).values,
                                            dtype=np.float64)
                        if scale != 1.0:
                            values = values * scale
                        valid &= np.isfinite(values)
                        pair.append(values)
                    pairs.append(pair)
                bn = valid.sum(axis=0)
                for k, (truth, prediction) in enumerate(pairs):
                    merge_moments(n, mean_t[k], mean_p[k], m2_t[k], m2_p[k], cov[k], truth, prediction, valid)
                    error = np.where(valid, prediction - truth, 0.0)
                    sse[k] += np.sum(error * error, axis=0)
                    sae[k] += np.sum(np.abs(error), axis=0)
                    truth_min[k] = np.minimum(truth_min[k], np.min(np.where(valid, truth, np.inf), axis=0))
                    truth_max[k] = np.maximum(truth_max[k], np.max(np.where(valid, truth, -np.inf), axis=0))
                n += bn
                del pairs, pair, values, error
                if start == 0 or stop == len(common_times) or stop % (BLOCK_SIZE * 100) == 0:
                    print(f'{component}: {stop:,}/{len(common_times):,} 时次', flush=True)
        enough = n >= 3
        for k, (group, lead) in enumerate(CONFIGS):
            gi, hi = GROUPS.index(group), LEADS.index(lead)
            where = (gi, hi, ci)
            mse = np.divide(sse[k], n, out=np.full(spatial_shape, np.nan), where=enough)
            rmse = np.sqrt(mse)
            mae = np.divide(sae[k], n, out=np.full(spatial_shape, np.nan), where=enough)
            denominator = np.sqrt(m2_t[k]) * np.sqrt(m2_p[k])
            correlation = np.divide(cov[k], denominator, out=np.full(spatial_shape, np.nan),
                                    where=enough & (denominator > 0))
            finite_r = np.isfinite(correlation)
            if np.any(np.abs(correlation[finite_r]) > 1.0 + 1e-10):
                raise ArithmeticError('相关系数超出数值舍入允许的[-1,1]范围。')
            correlation[finite_r] = np.clip(correlation[finite_r], -1.0, 1.0)
            truth_range = truth_max[k] - truth_min[k]
            truth_range = np.where(enough & np.isfinite(truth_range), truth_range, np.nan)
            rmspe = np.divide(rmse, truth_range, out=np.full(spatial_shape, np.nan),
                              where=enough & np.isfinite(truth_range) & (truth_range > 0))
            for name, value in zip(METRIC_NAMES, (correlation, rmse, mae, rmspe)):
                output[name][where] = value
            ranges[where] = truth_range
        counts[ci] = n
        del mean_t, mean_p, m2_t, m2_p, cov, sse, sae, truth_min, truth_max
    coords = dict(group=list(GROUPS), lead_hour=list(LEADS), component=list(COMPONENTS),
                  level=LEVELS, latitude=latitude, longitude=longitude)
    dims = ('group', 'lead_hour', 'component', 'level', 'latitude', 'longitude')
    ds = xr.Dataset({name: (dims, value) for name, value in output.items()}, coords=coords)
    ds['reference_range'] = (dims, ranges)
    ds['n_valid'] = (dims[2:], counts)
    for name in METRIC_NAMES:
        mask = np.all(np.isfinite(output[name]), axis=(0, 1))
        ds[f'common_spatial_mask_{name}'] = (dims[2:], mask.astype(np.uint8))
        ds[f'common_spatial_mask_{name}'].attrs['meaning'] = '1: this metric is finite in every group and lead'
    ds = ds.assign_coords(evaluation_time=('evaluation_time', common_times))
    ds.level.attrs['units'] = 'hPa'
    ds.latitude.attrs['units'] = 'degrees_north'
    ds.longitude.attrs['units'] = 'degrees_east'
    ds.R.attrs.update(units='1', long_name='Pearson correlation along common valid test times')
    ds.RMSPE.attrs.update(units='1', definition='RMSE / own_truth_temporal_range; no multiplication by 100')
    for name in ('RMSE', 'MAE', 'reference_range'):
        ds[name].attrs['component_units'] = 'u,v: m s-1; w component represents omega: Pa s-1'
    ds.attrs.update(schema_version=SCHEMA_VERSION, input_signature=input_signature,
                    common_count=int(len(common_times)), common_start=str(common_times[0]), common_end=str(common_times[-1]),
                    w_to_omega=float(W_TO_OMEGA), source_prediction='Final B is complete corrected forecast, not bare residuals.',
                    mask_definition='All groups/leads share finite time mask separately per component/level/grid.',
                    aggregation='Metric-specific common spatial masks; unweighted grid means; equal nine-unit means for R/RMSPE only.',
                    created_utc=datetime.now(timezone.utc).isoformat())
    assert_sources_unchanged()
    return ds


In [ ]:
BOUNDARY_PATH = None
BASIN_MASK_PATH = None           # 可选二维NC，变量basin_mask（1=流域内），经纬度必须精确匹配
BASIN_MASK_VARIABLE = 'basin_mask'
MAP_AGGREGATION = 'all_leads'      # 'all_leads': 6时效×3层等权；'6h': 6h×3层等权
VECTOR_LEAD = 6
ARROWS_ACROSS = 16
FIGSIZE_MAP = (10.8, 6.8)
FIGSIZE_VECTOR = (10.8, 9.4)
plt.rcParams.update({'font.family':'DejaVu Sans', 'font.size':14*FONT_SCALE,
                     'axes.titlesize':15*FONT_SCALE, 'axes.labelsize':14*FONT_SCALE,
                     'xtick.labelsize':12*FONT_SCALE, 'ytick.labelsize':12*FONT_SCALE,
                     'pdf.fonttype':42, 'ps.fonttype':42, 'savefig.facecolor':'white'})


def spatial_context():
    mask = np.ones((len(latitude),len(longitude)), dtype=bool)
    boundary = None
    if BOUNDARY_PATH is not None:
        import geopandas as gpd
        import shapely
        boundary = gpd.read_file(BOUNDARY_PATH)
        if boundary.crs is None:
            raise ValueError('流域边界缺少CRS，须先确定坐标系。')
        boundary = boundary.to_crs(4326)
        geom = boundary.geometry.union_all() if hasattr(boundary.geometry,'union_all') else boundary.geometry.unary_union
        xx,yy = np.meshgrid(longitude,latitude)
        mask &= np.asarray(shapely.covers(geom,shapely.points(xx,yy)),bool)
    if BASIN_MASK_PATH is not None:
        with xr.open_dataset(BASIN_MASK_PATH) as d:
            m=d[BASIN_MASK_VARIABLE]
            ren={a:b for b in ('latitude','longitude') for a in ALIASES[b] if a in m.dims and a!=b}
            m=m.rename(ren).transpose('latitude','longitude').sortby('latitude').sortby('longitude')
            if not np.array_equal(m.latitude,latitude) or not np.array_equal(m.longitude,longitude):
                raise ValueError('流域mask网格不匹配，禁止自动插值。')
            mask &= np.isfinite(m.values)&(m.values>0)
    if not mask.any(): raise ValueError('流域mask没有有效格点。')
    if boundary is None and BASIN_MASK_PATH is None:
        warnings.warn('未指定流域边界/掩膜：使用NC有效格点；若NC包含流域外数据，请设置BOUNDARY_PATH或BASIN_MASK_PATH。')
    return mask,boundary
def write_nc(ds,path):
    with tempfile.TemporaryDirectory(prefix='wind_nc_',dir=NETCDF_TEMP_DIR) as folder:
        tmp=Path(folder)/'data.nc'
        ds.to_netcdf(tmp,engine='netcdf4',encoding={k:{'zlib':True,'complevel':4} for k in ds.data_vars})
        shutil.copyfile(tmp,path)
def make_maps(ds, names):
    if MAP_AGGREGATION not in ('all_leads','6h'): raise ValueError('MAP_AGGREGATION只能为all_leads或6h。')
    chosen=ds.sel(group='B',lead_hour=list(LEADS) if MAP_AGGREGATION=='all_leads' else [6])
    maps={m:chosen[m].mean(('lead_hour','level'),skipna=False) for m in names}
    return xr.Dataset({m:maps[m].where(xr.DataArray(basin_mask,dims=('latitude','longitude'),coords={'latitude':latitude,'longitude':longitude})) for m in names})
def vector_means():
    shape=(len(LEVELS),len(latitude),len(longitude))
    totals=np.zeros((4,)+shape); n=np.zeros(shape,dtype=np.int64)
    keys=[('B',VECTOR_LEAD,c,k) for k in KINDS for c in ('u','v')]
    with ExitStack() as stack:
        fs=[canonical_field(stack.enter_context(xr.open_dataset(records[key]['path'],cache=False)),f'{key[3]}_{key[2]}',records[key]['path']) for key in keys]
        for start in range(0,len(common_times),BLOCK_SIZE):
            stop=min(start+BLOCK_SIZE,len(common_times))
            a=[np.asarray(f.isel(time=time_selection(records[key]['indices'][start:stop])).values,dtype=float) for key,f in zip(keys,fs)]
            good=np.logical_and.reduce([np.isfinite(v) for v in a])&basin_mask[None,None,:,:]
            for j,v in enumerate(a): totals[j]+=np.where(good,v,0).sum(axis=0)
            n+=good.sum(axis=0)
    a=np.divide(totals,n[None],out=np.full(totals.shape,np.nan),where=n[None]>0)
    return a,n


In [ ]:
records, common_times, latitude, longitude, input_signature, source_info = scan_inputs()
basin_mask, boundary = spatial_context()
metric_fields = compute_grid_metrics()
plotted = make_maps(metric_fields, METRIC_NAMES)
write_nc(plotted, METRICS_DIR / 'plotted_spatial_metrics.nc')
a, n = vector_means()
d=xr.Dataset({name:(('level','latitude','longitude'),v) for name,v in zip(('test_u','test_v','pred_u','pred_v'),a)},coords={'level':LEVELS,'latitude':latitude,'longitude':longitude})
d['difference_u']=d.test_u-d.pred_u;d['difference_v']=d.test_v-d.pred_v
d['n_valid']=(('level','latitude','longitude'),n)
d.attrs.update(forecast_lead_hour=VECTOR_LEAD,definition='Temporal mean vector, NOT mean scalar wind speed',time_start=str(common_times[0]),time_end=str(common_times[-1]))
write_nc(d,METRICS_DIR/'mean_horizontal_wind_6h.nc')
assert_sources_unchanged()
